# Notebook 05: Hybrid Allergen Detection

## Overview

Combines rule-based and ML predictions for robust allergen detection.

## Workflow

1. Load pre-trained model using utility modules
2. Load hybrid configuration
3. Define rule-based patterns (import from text_processing)
4. Hybrid prediction logic
5. Evaluate on test set using utility evaluation functions
6. Save configuration

## 0. Setup & Imports

Using modular utility functions to reduce code duplication

In [3]:
import os
import sys
import json
import ast
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score
from scipy.special import expit

# Import project utility modules
from utils.data_utils import (
    load_labeled_data,
    create_stratified_splits,
    get_data_directories
)
from utils.text_processing import (
    rule_match,
    get_allergen_list,
    clean_html,
    parse_label_string,
    allergens_to_binary
)
from utils.model_utils import (
    load_model_and_tokenizer,
    load_hybrid_config,
    predict_ml,
    hybrid_predict,
    find_best_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis
)

# Set seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Imports completed successfully using utility modules!")

Imports completed successfully using utility modules!


## Setup Complete

All imports consolidated in the cell above, reducing code duplication across the notebook.

In [4]:
# Define data and config paths
DATA_PATH = "../data/final/labeled_dataset_enhanced.csv"
BEST_THRESHOLDS_PATH = "../models/best_thresholds.npy"
HYBRID_CONFIG_PATH = "../models/hybrid_config.json"

# Load labeled data using utility function
df = load_labeled_data(DATA_PATH)

# Create clean_text from ingredients_text_en (same as notebook 04)
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Parse labels using utility function
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

Dataset shape: (1057, 13)
Label distribution (per class): [497  98  88  16 277 355  77  32]


In [5]:
# Generate stratified train/val/test splits (same approach as notebook 04)
# using the utility function from utils.data_utils

X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {(np.array(train_labels).sum(axis=1)>0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {(np.array(val_labels).sum(axis=1)>0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {(np.array(test_labels).sum(axis=1)>0).sum()})")

Train size: 739 (positive samples: 486)
Val size:   159   (positive samples: 120)
Test size:  159  (positive samples: 106)


## 3. Rule‑Based System

Using the robust rule_match function from text_processing utility

In [6]:
# The rule_match function is imported from utils.text_processing
# It includes comprehensive allergen keywords and negation handling
# No need to redefine it here - using the utility version ensures consistency

# Example usage:
# rule_present = rule_match(text, "milk")
print("Using rule_match from utils.text_processing")
allergen_list = get_allergen_list()
print(f"Available allergens: {allergen_list}")

Using rule_match from utils.text_processing
Available allergens: ['milk', 'eggs', 'peanuts', 'tree_nuts', 'soy', 'wheat', 'fish', 'shellfish']


## 4. Load the Trained MobileBERT Model and Tokenizer

Using utility functions for model loading

In [7]:
# Load the trained MobileBERT model and tokenizer using utilities
MODEL_PATH = "../models/mobilebert_allergen_final/"

print(f"Loading model from {MODEL_PATH}...")
model, tokenizer, device = load_model_and_tokenizer(MODEL_PATH)

# Load hybrid configuration (path already set in cell 6)
hybrid_config = load_hybrid_config(HYBRID_CONFIG_PATH)

ml_thresholds = hybrid_config["ml_thresholds"]
rule_conf_threshold = hybrid_config["rule_conf_threshold"]
mode = hybrid_config["mode"]

print(f"Loaded ML thresholds: {ml_thresholds}")
print(f"Rule conf threshold: {rule_conf_threshold}")
print(f"Mode: {mode}")

Loading model from ../models/mobilebert_allergen_final/...


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

Loaded ML thresholds: [0.08, 0.5700000000000001, 0.04, 0.01, 0.17, 0.03, 0.01, 0.03]
Rule conf threshold: 0.5
Mode: rule_priority


## 5. Text Preprocessing Function

Simple preprocessing (can be enhanced with utilities if needed)

In [8]:
# Text preprocessing function (simple, kept here for notebook clarity)
def preprocess_text(text):
    return text.lower().strip()

## 6. Load Optimal ML Thresholds

Using utility function to find optimal thresholds if not already saved

In [9]:
# Try to load pre-computed thresholds
try:
    best_thresholds = np.load(BEST_THRESHOLDS_PATH)
    print(f"Loaded pre-computed thresholds: {best_thresholds}")
except FileNotFoundError:
    print("Pre-computed thresholds not found. Computing from validation set...")
    # Get validation predictions
    val_ml_preds, val_probs = predict_ml(val_texts, model, tokenizer, device, max_length=209)
    best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
    print("Computed thresholds:", best_thresholds)

print(f"Using thresholds: {best_thresholds}")

Loaded pre-computed thresholds: [0.08 0.57 0.04 0.01 0.17 0.03 0.01 0.03]
Using thresholds: [0.08 0.57 0.04 0.01 0.17 0.03 0.01 0.03]


## 7. Evaluate on Test Set

Using utility evaluation functions for consistent reporting

In [10]:
# ML only baseline
ml_test_preds, _ = predict_ml(test_texts, model, tokenizer, device, thresholds=best_thresholds)
print("\n=== ML ONLY (test set) ===")
print_classification_report(test_labels, ml_test_preds, get_allergen_list(), prefix="ML Only")

# Hybrid (hard override with global optimal threshold)
test_hybrid_preds, _ = hybrid_predict(test_texts, model, tokenizer, device,
                                      thresholds=best_thresholds,
                                      rule_conf_threshold=rule_conf_threshold,
                                      mode='hard_override')
print("\n=== HYBRID (hard override) ===")
print_classification_report(test_labels, test_hybrid_preds, get_allergen_list(), prefix="Hybrid (hard override)")

# Try alternative mode: high confidence bypass
test_hybrid_bypass, _ = hybrid_predict(test_texts, model, tokenizer, device,
                                       thresholds=best_thresholds,
                                       rule_conf_threshold=rule_conf_threshold,
                                       mode='high_confidence_bypass')
print("\n=== HYBRID (high confidence bypass) ===")
print_classification_report(test_labels, test_hybrid_bypass, get_allergen_list(), prefix="Hybrid (high confidence bypass)")


=== ML ONLY (test set) ===
=== ML Only ===
              precision    recall  f1-score   support

        milk     0.9595    0.9467    0.9530        75
        eggs     1.0000    0.9286    0.9630        14
     peanuts     0.8571    0.9231    0.8889        13
   tree_nuts     0.4000    0.6667    0.5000         3
         soy     0.8889    0.9524    0.9195        42
       wheat     0.9636    0.9815    0.9725        54
        fish     0.7692    0.9091    0.8333        11
   shellfish     0.8333    1.0000    0.9091         5

   micro avg     0.9156    0.9493    0.9321       217
   macro avg     0.8340    0.9135    0.8674       217
weighted avg     0.9230    0.9493    0.9348       217
 samples avg     0.6231    0.6329    0.6236       217

Macro F1: 0.8674
Micro F1: 0.9321


=== HYBRID (hard override) ===
=== Hybrid (hard override) ===
              precision    recall  f1-score   support

        milk     0.9595    0.9467    0.9530        75
        eggs     1.0000    0.9286    0.9630 

## 8. Error Analysis (examples where hybrid differs from ML)

Using utility function for error analysis

In [11]:
results_df = pd.DataFrame({
    "text": test_texts,
    "true": [list(l) for l in test_labels],
    "ml_pred": [list(p) for p in ml_test_preds],
    "hybrid_pred": [list(p) for p in test_hybrid_preds]
})
differences = results_df[results_df["ml_pred"] != results_df["hybrid_pred"]]
print(f"Number of test samples where hybrid changed prediction: {len(differences)}")
if len(differences) > 0:
    print("\nFirst 3 examples:")
    allergen_list = get_allergen_list()
    for i in range(min(3, len(differences))):
        row = differences.iloc[i]
        true_all = [allergen_list[j] for j, v in enumerate(row['true']) if v == 1]
        ml_all = [allergen_list[j] for j, v in enumerate(row['ml_pred']) if v == 1]
        hybrid_all = [allergen_list[j] for j, v in enumerate(row['hybrid_pred']) if v == 1]
        print(f"\nText: {row['text'][:150]}...")
        print(f"True: {true_all}")
        print(f"ML:   {ml_all}")
        print(f"Hybrid:{hybrid_all}")

Number of test samples where hybrid changed prediction: 0


## 9. Save hybrid configuration for production

Saving the current hybrid configuration

In [12]:
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": rule_conf_threshold,
    "mode": mode
}
with open("../models/hybrid_config.json", "w") as f:
    json.dump(hybrid_config, f, indent=2)
print("Hybrid configuration saved to ../models/hybrid_config.json")

Hybrid configuration saved to ../models/hybrid_config.json
